## The Rise of "Thinking" Models
Models like OpenAI o1/GPT-5.5 extended thinking, DeepSeek-R2, and Claude Opus 4.7 have CoT "baked in" via Reinforcement Learning (RL).

- System-Level CoT: The model doesn't just "print" reasoning; it has a dedicated "Thinking Window."
- Hidden CoT: In many enterprise versions, the reasoning chain is hidden from the user but verifiable by the system to prevent prompt injection or "thought leakage."
- Scaling Law: These models follow the Inference Scaling Law—the longer they "think," the better they solve hard problems (
o
1
 can solve gold-medal IMO math given enough time).

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key="YOUR_API_KEY")

response = client.models.generate_content(
    model="gemini-2.5-pro",
    contents="Solve: If a train travels 120 km in 2 hours, what is its speed?",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_budget=2048
        )
    )
)

print(response.text)

# Self-Correction and Verification
Production pipelines no longer trust a single Chain-of-Thought. They layer in Self-Verification.

# Process
1. Generate Answer A via CoT.
2. Critique: "Are there any errors in the logic above?"
3. If errors: "Correct the logic and provide Answer B."


Nuance: This is now integrated into Execution-Verified CoT for coding, where the model writes the logic, runs the code, and corrects itself if the code fails.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# -----------------------------
# STEP 1: Generate Answer
# -----------------------------
generate_prompt = ChatPromptTemplate.from_template("""
You are an expert problem solver.

Question:
{question}

Instructions:
1. Think step-by-step.
2. Show your reasoning.
3. Produce an answer.

Format:

REASONING:
...

ANSWER:
...
""")

generate_chain = generate_prompt | llm

# -----------------------------
# STEP 2: Critique Answer
# -----------------------------
critique_prompt = ChatPromptTemplate.from_template("""
You are a strict reviewer.

Review the following solution.

Question:
{question}

Solution:
{solution}

Tasks:
1. Identify logical errors.
2. Identify calculation errors.
3. Identify missing assumptions.
4. State whether the answer is correct.

Format:

CRITIQUE:
...

IS_CORRECT:
Yes/No
""")

critique_chain = critique_prompt | llm

# -----------------------------
# STEP 3: Refine Answer
# -----------------------------
refine_prompt = ChatPromptTemplate.from_template("""
You are a senior expert.

Question:
{question}

Original Solution:
{solution}

Critique:
{critique}

Instructions:
1. Fix all identified issues.
2. Re-solve if necessary.
3. Produce the final corrected answer.

Format:

CORRECTED_REASONING:
...

FINAL_ANSWER:
...
""")

refine_chain = refine_prompt | llm

# -----------------------------
# RUN PIPELINE
# -----------------------------
question = """
A train travels 120 km in 2 hours.
What is its speed?
"""

# Generate
solution = generate_chain.invoke({
    "question": question
})

print("\n===== GENERATED =====\n")
print(solution.content)

# Critique
critique = critique_chain.invoke({
    "question": question,
    "solution": solution.content
})

print("\n===== CRITIQUE =====\n")
print(critique.content)

# Refine
final_answer = refine_chain.invoke({
    "question": question,
    "solution": solution.content,
    "critique": critique.content
})

print("\n===== FINAL =====\n")
print(final_answer.content)

# When CoT Fails (Over-thinking)

CoT is not a silver bullet. For simple tasks, it adds:

- Latency: More tokens = slower response.
- Cost: You pay for every "thought" token.
- Over-thinking: The model might hallucinate complexity where none exists (e.g., explaining why 2+2=4 for 3 paragraphs).